# Goodput-Analyse für OmniSphinx und native Mix-Formate

Dieses Notebook untersucht die Goodput-Effizienz verschiedener Mix-Format-Varianten.
Die betrachteten Varianten umfassen OmniSphinx in unterschiedlichen Emulationsmodi sowie native Sphinx- und PolySphinx-Formate.
Für alle Analysen setzen wir die Pfadlänge auf $r = 5$ und verwenden die in der Implementierung hinterlegte Sicherheitsparameterisierung mit $\kappa = 16$ Bytes.


## Aufgabenstellung

1. **Beta-Länge vs. Goodput**  
   * Fixierte Payload-Länge: 512 Bytes.  
   * Variiere die Beta-Länge (nur gültige Werte) und berechne Goodput sowie Paketgröße für jede Variante.  
   * Visualisiere Goodput in Abhängigkeit der Beta-Länge.

2. **Payload-Größe vs. Goodput**  
   * Fixiere die Header-Größen (Alpha, Beta, Gamma) pro Variante.  
   * Variiere die Payload-Größe von 0 bis 4096 Bytes und berechne den Goodput.  
   * Visualisiere den Goodput in Abhängigkeit der Payload-Größe.

Die Beta-Längen für die OmniSphinx-Varianten werden gemäß den bereitgestellten Formeln berechnet.
Für native Varianten orientieren wir uns an den Referenzformeln aus der Literatur bzw. der Java-Implementierung.


In [1]:
import math
from dataclasses import dataclass
from typing import Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')


## Parameter und Hilfsfunktionen

Die folgenden Konstanten basieren auf der Standard-Parametrisierung aus der Codebasis:

* Sicherheitsparameter (MAC-Länge) $\kappa = 16$ Bytes.
* Elliptische Kurve **secp224r1** ⇒ unkomprimierte Punktdarstellung (Alpha) mit $1 + 2 \cdot 28 = 57$ Bytes.
* Gamma entspricht einem MAC mit Länge $\kappa$.
* Pfadlänge $r = 5$ Hops.

Zusätzlich werden Hilfsfunktionen definiert, um Beta-Längen und Goodput zu berechnen.


In [ ]:
BITS_PER_BYTE = 8
KAPPA = 128  #Bits
ALPHA_LEN = 2*KAPPA
GAMMA_LEN = KAPPA
PATH_LENGTH = 5

def OmniSphinx_PolySphonx_Relay_Instruction_Length(kappa: int = KAPPA) -> int:
    """Return the encoded length of a PolySphinx relay instruction block in bits."""

    load_sigma = (
        kappa  # sigma register payload in bits
        + 3 * BITS_PER_BYTE  # opcode, length field, and target register (1 byte each)
    )
    encrypt = 4 * BITS_PER_BYTE  # opcode + key register + input register + output register
    load_next_hop = (
        kappa  # next-hop data length in bits
        + 3 * BITS_PER_BYTE  # opcode, length field, target register
    )
    forward = 2 * BITS_PER_BYTE  # opcode + sigma register identifier
    raw_len = load_sigma + encrypt + load_next_hop + forward  # sum the instruction fragments
    return raw_len + BITS_PER_BYTE  # leading length byte in the bytecode stream

def OmniSphinx_PolySphonx_Exit_Instruction_Length(p: int, r: int, *, kappa: int = KAPPA) -> int:
    """Return the encoded length of an exit instruction block for PolySphinx in bits."""

    log2p = math.ceil(math.log2(p))
    # Each hop contributes ``log2p`` selector bits, but at least one byte is emitted per hop.
    path_len = log2p * (r + 1)

    load_seed = (kappa  # shared seed payload length
        + 3 * BITS_PER_BYTE  # opcode + length byte + destination register
    )
    load_path = (
        path_len  # hop selector bits across all remaining hops
        + 3 * BITS_PER_BYTE  # opcode + length byte + destination register
    )
    load_recipient = (
        kappa  # recipient public key length in bits
        + 3 * BITS_PER_BYTE  # opcode + length byte + destination register
    )
    hash_seed = 3 * BITS_PER_BYTE  # opcode + input register + output register
    append_seed = 4 * BITS_PER_BYTE  # opcode + loop register + input register + output register
    loop_setup = 3 * BITS_PER_BYTE  # opcode + loop register + counter register
    loop_body = (
        4 * BITS_PER_BYTE  # load path fragment (opcode + len + register + length register)
        + 4 * BITS_PER_BYTE  # concatenate path piece (opcode + register trio)
        + 3 * BITS_PER_BYTE  # compute hash (opcode + input + output)
        + 3 * BITS_PER_BYTE  # xor with seed (opcode + input + output)
        + 4 * BITS_PER_BYTE  # append recipient data (opcode + register trio)
    )
    decrypt_loop = 3 * BITS_PER_BYTE  # opcode + loop counter register + limit register
    decrypt_body = (
        4 * BITS_PER_BYTE  # decrypt call (opcode + key + cipher + output registers)
        + 4 * BITS_PER_BYTE  # forward decrypted block (opcode + register trio)
    )
    forward = 2 * BITS_PER_BYTE  # opcode + payload register

    raw_len = (
        load_seed
        + load_path
        + load_recipient
        + hash_seed
        + append_seed
        + loop_setup
        + loop_body
        + decrypt_loop
        + decrypt_body
        + forward
    )
    return raw_len + BITS_PER_BYTE  # instruction prefix encoding the total length

def OmniSphinx_PolySphonx_Replication_Instruction_Length(p: int,tau_post: int,kappa: int = KAPPA) -> int:
    """Return the encoded length of the replication instruction block in bits."""

    if tau_post < 0:
        raise ValueError("Post-replication beta length must be >= 0")

    subheader_len = (
        p
        * (
            5 * kappa  # five per-path values (public keys, MACs, etc.)
            + tau_post  # beta suffix for the replicated packet
        )
    )
    load_subheaders = (
        subheader_len  # aggregate replicated subheader bits
        + 3 * BITS_PER_BYTE  # opcode + length byte + destination register
    )
    loop_setup = 3 * BITS_PER_BYTE  # opcode + loop register + limit register
    loop_body = (
        4 * BITS_PER_BYTE  # compute MAC (opcode + three registers)
        + 4 * BITS_PER_BYTE  # derive key (opcode + three registers)
        + 4 * BITS_PER_BYTE  # load sigma (opcode + three registers)
        + 4 * BITS_PER_BYTE  # encrypt sigma (opcode + three registers)
        + 4 * BITS_PER_BYTE  # load beta suffix (opcode + three registers)
        + 4 * BITS_PER_BYTE  # concatenate headers (opcode + three registers)
        + 2 * BITS_PER_BYTE  # branch-and-forward (opcode + loop register)
    )
    raw_len = load_subheaders + loop_setup + loop_body  # combine static and loop costs
    return raw_len + BITS_PER_BYTE  # encoded instruction length prefix

def sphinx_instruction_length(kappa: int = KAPPA) -> int:
    """Length of a single OmniSphinx instruction block (including length byte) in bits."""

    forward_len = (
        BITS_PER_BYTE  # opcode identifying the forward primitive
        + BITS_PER_BYTE  # length of the payload in bytes encoded as one byte
        + kappa  # payload bits moved into the sigma register
    )
    raw_len = (
        3 * BITS_PER_BYTE  # concatWithByteValue: opcode + target register + literal byte
        + 3 * BITS_PER_BYTE  # hash: opcode + input register + output register
        + 4 * BITS_PER_BYTE  # decrypt: opcode + key + ciphertext + output registers
        + forward_len  # forward invocation with payload
        + 2 * BITS_PER_BYTE  # mixing primitive: opcode + register selector
    )
    return raw_len


def OmniSphinx_polysphinx_post(p: int, r: int, kappa: int = KAPPA) -> int:
    """Return beta length for the post-replication suffix paths in PolySphinx (bits)."""

    relay_len = OmniSphinx_PolySphonx_Relay_Instruction_Length(kappa=kappa)
    exit_len = OmniSphinx_PolySphonx_Exit_Instruction_Length(p, r, kappa=kappa)
    return r * relay_len + exit_len


def OmniSphinx_polysphinx_pre(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Return beta length before replication for native PolySphinx (bits)."""

    tau_post = tau_polysphinx_post(p, r=r - 1, kappa=kappa)
    return (
        (r - 1) * (BITS_PER_BYTE + 3 * kappa)
        + BITS_PER_BYTE
        + p * (5 * kappa + tau_post)
    )

def polysphinx_post(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Beta-Länge (Post-Replikation) für OmniSphinx-PolySphinx."""
    return r * (8 + 3 * kappa) + kappa + r * math.ceil(math.log2(p)) 


def polysphinx_pre(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Beta-Länge (Pre-Replikation) für native PolySphinx."""
    tau_post = tau_polysphinx_post(p, r=r, kappa=kappa)
    return (r - 1) * (8 + 3 * kappa) + 8 + p * (5 * kappa + tau_post) 


def beta_native_sphinx(r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Return beta length for native Sphinx (no instruction overhead) in bits."""

    return r * (2 * kappa + BITS_PER_BYTE)


def tau_polysphinx_omni(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Return beta length for OmniSphinx running in PolySphinx emulation mode (bits)."""

    tau_post = tau_polysphinx_post(p, r=r - 1, kappa=kappa)
    replication_instr_len = OmniSphinx_PolySphonx_Replication_Instruction_Length(
        p, tau_post, kappa=kappa
    )
    return replication_instr_len + kappa + BITS_PER_BYTE


def tau_sphinx_omni(r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Return beta length for OmniSphinx running in Sphinx emulation mode (bits)."""

    instr_len = sphinx_instruction_length(kappa)
    return r * (instr_len + kappa + BITS_PER_BYTE)


def packet_size(payload_len: int,beta_len: int,alpha_len: int = ALPHA_LEN,gamma_len: int = GAMMA_LEN) -> int:
    """Compute the total packet size for the given payload and header lengths (bits)."""

    return alpha_len + beta_len + gamma_len + payload_len


def goodput(payload_len: int, total_len: int) -> float:
    """Goodput defined as the ratio of payload bits to transmitted bits."""

    if total_len == 0:
        return 0.0
    return payload_len / total_len

	
def goodput_unicast(payload_len: int, total_len: int, p:int) -> float:
    """Goodput-Wenn der Sender um Multicast zu emulieren einfach mehrere unicast nachrichten schickt."""
    if total_len == 0:
        return 0.0
    return payload_len / (total_len * p)


@dataclass(frozen=True)
class Variant:
    name: str
    beta_min: int
    kind: str
    p: Optional[int] = None


variants: Dict[str, Variant] = {
    "omni_sphinx": Variant("OmniSphinx – Sphinx (emuliert)", tau_sphinx_omni(), "omni"),
    "omni_poly_p3": Variant("OmniSphinx – Poly (p=3)", tau_polysphinx_pre(3), "omni", 3),
    "omni_poly_p5": Variant("OmniSphinx – Poly (p=5)", tau_polysphinx_pre(5), "omni", 5),
    "omni_poly_p10": Variant("OmniSphinx – Poly (p=10)", tau_polysphinx_pre(10), "omni", 10),
    "sphinx_native": Variant("Sphinx (nativ)", beta_native_sphinx(), "native"),
    "poly_native_p3": Variant("PolySphinx (nativ, p=3)", tau_polysphinx_pre(3), "native", 3),
    "poly_native_p5": Variant("PolySphinx (nativ, p=5)", tau_polysphinx_pre(5), "native", 5),
    "poly_native_p10": Variant("PolySphinx (nativ, p=10)", tau_polysphinx_pre(10), "native", 10),
}

summary_df = pd.DataFrame(
    {
        "Variante": [v.name for v in variants.values()],
        "βₘᵢₙ [B]": [v.beta_min for v in variants.values()],
        "Replikationsfaktor": [v.p if v.p is not None else "–" for v in variants.values()],
    }
)
summary_df


,Variante,βₘᵢₙ [B],Replikationsfaktor
0,OmniSphinx – Sphinx (emuliert),235,–
1,OmniSphinx – Poly (p=3),1215,3
2,OmniSphinx – Poly (p=5),1897,5
3,OmniSphinx – Poly (p=10),3602,10
4,Sphinx (nativ),165,–
5,"PolySphinx (nativ, p=3)",1215,3
6,"PolySphinx (nativ, p=5)",1897,5
7,"PolySphinx (nativ, p=10)",3602,10
